# InferenceGuard: Rewriter Generalization via Teacher Distillation
**Task 1**: Distill complex reasoning from teacher models into a lightweight student model (`Qwen2.5-0.5B-Instruct` / `Qwen2.5-1.5B-Instruct`), persist checkpoints directly to Google Drive, and evaluate generalization against the held-out test split (`data/splits/test.jsonl`).

### Objectives:
1. **Teacher Distillation**: Generate diverse synthetic rewriting demonstrations covering varied tones, complex syntactic clauses, and edge cases with zero bracketed placeholders (`[LOCATION]`, etc.).
2. **Student QLoRA Fine-Tuning**: Fine-tune the small student model with completion-only LM loss masking (`DataCollatorForCompletionOnlyLM`) on ChatML prompts.
3. **Google Drive Fault-Tolerance**: Sync intermediate checkpoints and final model weights directly to mounted Google Drive to survive Colab preemptions/disconnections.
4. **Generalization Validation**: Prove that the distilled student model resolves out-of-distribution generalization failures on `data/splits/test.jsonl` compared to the static prompt baseline (`src/rewriter/prompt_baseline.py`), achieving higher utility preservation and near-zero degeneracy.

## 1. Environment Setup & Branch Checkout
Ensure the repository is cloned and switch to `feature/rewriter-teacher-distillation` containing the distillation dataset, Google Drive callback, and utility evaluation harness.

In [ ]:
# Cell 1: Environment Setup, Branch Checkout & Repository Verification
import os
from pathlib import Path

# Ensure we are in the Inference-Guard directory
if not Path("src/rewriter/train_sft.py").exists():
    if Path("/content/Inference-Guard").exists():
        %cd /content/Inference-Guard
    elif Path("/content/drive/MyDrive/Inference-Guard").exists():
        %cd /content/drive/MyDrive/Inference-Guard
    else:
        %cd /content
        !git clone https://github.com/GIND123/Inference-Guard.git
        %cd Inference-Guard

# Fetch and checkout the distillation feature branch
!git fetch origin feature/rewriter-teacher-distillation
!git checkout -B feature/rewriter-teacher-distillation origin/feature/rewriter-teacher-distillation
!git pull origin feature/rewriter-teacher-distillation

print("\n--- Current Git Status ---")
!git branch -v
!git log -1 --oneline


## 2. Mount Google Drive for Fault-Tolerant Checkpoint Persistence
Mount Google Drive so intermediate training steps and final LoRA adapter weights are saved directly to persistent storage. If your Colab session times out or disconnects, your weights are safely retained.

In [ ]:
# Cell 2: Mount Google Drive (with automatic credential handshake & fallback)
from pathlib import Path

drive_mounted = False
drive_05b_dir = Path("/content/drive/MyDrive/InferenceGuard/rewriter_0.5b")
drive_15b_dir = Path("/content/drive/MyDrive/InferenceGuard/rewriter_1.5b")

try:
    from google.colab import auth, drive
    try:
        # Explicit auth handshake bypasses browser iframe token propagation errors
        auth.authenticate_user()
    except Exception:
        pass

    drive.mount('/content/drive', force_remount=True)
    drive_05b_dir.mkdir(parents=True, exist_ok=True)
    drive_15b_dir.mkdir(parents=True, exist_ok=True)
    drive_mounted = True
    print(f"✓ Google Drive successfully mounted! Checkpoints will persist to:\n - {drive_05b_dir}\n - {drive_15b_dir}")
except Exception as err:
    print("=" * 70)
    print("⚠ Notice: Google Drive mounting could not be completed.")
    print(f"Detail: {err}")
    print("\nCommon fixes if you want Google Drive backup:")
    print(" 1. Multiple Google Accounts: Log into only one account in this browser, or use Incognito.")
    print(" 2. Browser Extensions: Temporarily disable Brave Shields / ad blockers for colab.research.google.com.")
    print(" 3. Left-Panel Icon: Click the 'Files' folder icon on the left sidebar, then click the 'Mount Drive' icon.")
    print("\n→ Proceeding in local storage mode. Checkpoints and weights will save directly to:")
    print("   artifacts/rewriter_qlora_0.5b/ (no work is lost during this active session).")
    print("=" * 70)


## 3. Install Dependencies & Inspect GPU Environment
Install the required training and evaluation libraries (`transformers`, `peft`, `trl`, `datasets`, `bitsandbytes`, `accelerate`).

In [ ]:
# Cell 3: Install dependencies & inspect GPU
# Remove unused multimodal packages that trigger CUDA binary mismatches in Colab
!pip uninstall -y -q torchvision torchaudio
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")


## 4. Dataset Inspection & Teacher Distillation Data Verification
Check that `artifacts/rewriter_sft_chatml.jsonl` and held-out test split `data/splits/test.jsonl` are present.

In [ ]:
# Cell 4: Verify or regenerate teacher distillation training data
import json
from pathlib import Path

train_path = Path("artifacts/rewriter_sft_chatml.jsonl")
test_path = Path("data/splits/test.jsonl")

assert test_path.exists(), f"Held-out test split missing: {test_path}"
test_count = sum(1 for _ in open(test_path, encoding='utf-8'))
print(f"Held-out test split: {test_path} ({test_count} samples)")

# Regenerate training data to incorporate bidirectional persona pruning and clean generalizations
# Set to True to rebuild the dataset with the updated teacher generator
FORCE_REGENERATE = True

if FORCE_REGENERATE or not train_path.exists():
    print("Generating diverse teacher distillation dataset from data/splits/train.jsonl...")
    !python -m src.rewriter.generate_training_data \
        --input data/splits/train.jsonl \
        --output artifacts/rewriter_sft_chatml.jsonl \
        --output_alpaca artifacts/rewriter_sft_alpaca.jsonl \
        --max_samples 500

train_count = sum(1 for _ in open(train_path, encoding='utf-8'))
print(f"\nSFT Training dataset ready: {train_path} ({train_count} ChatML samples)")

# Preview one sample from training data
with open(train_path, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())
    print("\nSample ChatML Structure:")
    print(json.dumps(sample, indent=2)[:600] + "\n...")


## 5. Fine-Tune Lightweight Student Model (Qwen 0.5B)
Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` using QLoRA with 4-bit quantization and completion-only loss masking.
Intermediate checkpoints (`checkpoint-*`) and final adapter weights will sync directly to Google Drive.

In [ ]:
# Cell 5: Fine-tune Qwen 0.5B Student Model
# Proactively remove unused multimodal extensions that trigger CUDA mismatches in Colab
!pip uninstall -y -q torchvision torchaudio 2>/dev/null || true
from pathlib import Path
drive_flag = "--drive_path /content/drive/MyDrive/InferenceGuard/rewriter_0.5b" if Path("/content/drive/MyDrive").exists() else ""

!python -m src.rewriter.train_sft \
    --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \
    --output_dir artifacts/rewriter_qlora_0.5b \
    --train_file artifacts/rewriter_sft_chatml.jsonl \
    --epochs 3 \
    --batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --save_steps 50 \
    {drive_flag}


## 6. Evaluate 0.5B Student vs Static Prompt Baseline on Held-Out Test Split
Compare the fine-tuned student against the static prompt baseline (`src/rewriter/prompt_baseline.py`) on held-out test data (`data/splits/test.jsonl`).
Computes bidirectional entailment, contradiction probability, token edit ratio, degeneracy rate, and utility preservation.

In [ ]:
# Cell 6: Run utility evaluation on held-out test split
!python -m src.evaluation.utility_metrics \
    --test_split data/splits/test.jsonl \
    --adapter_dir artifacts/rewriter_qlora_0.5b \
    --base_model Qwen/Qwen2.5-0.5B-Instruct \
    --num_samples 50 \
    --compare_baseline \
    --output_report reports/heldout_utility_0.5b.json


In [ ]:
# Cell 7: Display and validate Qwen 0.5B generalization results
import json
from pathlib import Path

report_path = Path("reports/heldout_utility_0.5b.json")
if not report_path.exists():
    raise FileNotFoundError(f"Evaluation report not found at {report_path}. Please check Cell 6 execution logs above.")

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("=" * 75)
print("=== QWEN 0.5B HELD-OUT GENERALIZATION EVALUATION REPORT ===")
print("=" * 75)

baseline = report.get("static_prompt_baseline", {})
student = report.get("teacher_distilled_student", {})
delta = report.get("delta", {})

print(f"{'Metric':<38} | {'Static Baseline':<15} | {'Distilled Student':<17} | {'Delta':<10}")
print("-" * 88)
print(f"{'Bidirectional Entailment':<38} | {baseline.get('mean_bidirectional_entailment', 0):<15.4f} | {student.get('mean_bidirectional_entailment', 0):<17.4f} | {delta.get('bidirectional_entailment_gain', 0):+10.4f}")
print(f"{'Contradiction Prob (lower is better)':<38} | {baseline.get('mean_contradiction_prob', 0):<15.4f} | {student.get('mean_contradiction_prob', 0):<17.4f} | {delta.get('contradiction_drop', 0):+10.4f}")
print(f"{'Utility Preservation Rate':<38} | {baseline.get('utility_preservation_rate', 0):<15.4f} | {student.get('utility_preservation_rate', 0):<17.4f} | {delta.get('utility_preservation_gain', 0):+10.4f}")
print(f"{'Degeneracy Rate (lower is better)':<38} | {baseline.get('degeneracy_rate', 0):<15.4f} | {student.get('degeneracy_rate', 0):<17.4f} | {student.get('degeneracy_rate', 0) - baseline.get('degeneracy_rate', 0):+10.4f}")
print("-" * 88)
print(f"Generalization Out-of-Distribution Improved: {report.get('generalization_improved', False)}")


## 7. (Optional Quality Ceiling) Train & Evaluate Qwen 1.5B Model
Train and evaluate `Qwen2.5-1.5B-Instruct` as a higher-capacity ceiling comparison, saving checkpoints directly to `/content/drive/MyDrive/InferenceGuard/rewriter_1.5b`.

In [ ]:
# Cell 8: (Optional) Fine-tune and evaluate Qwen 1.5B Model
!pip uninstall -y -q torchvision torchaudio 2>/dev/null || true
from pathlib import Path
drive_flag_15b = "--drive_path /content/drive/MyDrive/InferenceGuard/rewriter_1.5b" if Path("/content/drive/MyDrive").exists() else ""

!python -m src.rewriter.train_sft \
    --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
    --output_dir artifacts/rewriter_qlora_1.5b \
    --train_file artifacts/rewriter_sft_chatml.jsonl \
    --epochs 3 \
    --batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --save_steps 50 \
    {drive_flag_15b}

!python -m src.evaluation.utility_metrics \
    --test_split data/splits/test.jsonl \
    --adapter_dir artifacts/rewriter_qlora_1.5b \
    --base_model Qwen/Qwen2.5-1.5B-Instruct \
    --num_samples 50 \
    --compare_baseline \
    --output_report reports/heldout_utility_1.5b.json


## 8. Promote Best Adapter to Production & Verify Readiness Gates
Promote the winning student model to `artifacts/rewriter_qlora` and register it with `scripts/check_eval_readiness.py` to satisfy Gate 8 evaluation readiness.

In [ ]:
# Cell 9: Promote adapter and verify Gate 8 readiness
import json
import shutil
import yaml
from pathlib import Path

# Select best performing student model (0.5B for fast inference, or 1.5B if higher capacity is desired)
source_adapter = Path("artifacts/rewriter_qlora_0.5b")
prod_adapter = Path("artifacts/rewriter_qlora")

if source_adapter.exists():
    prod_adapter.mkdir(parents=True, exist_ok=True)
    for item in source_adapter.glob("*"):
        if item.is_file():
            shutil.copy2(item, prod_adapter / item.name)
    print(f"Successfully promoted {source_adapter} -> {prod_adapter}")

    # Align experiment configuration with promoted student model base
    cfg_file = prod_adapter / "adapter_config.json"
    exp_file = Path("configs/experiments/session08.yaml")
    if cfg_file.exists() and exp_file.exists():
        adapter_data = json.loads(cfg_file.read_text(encoding="utf-8"))
        promoted_base = adapter_data.get("base_model_name_or_path", "Qwen/Qwen2.5-0.5B-Instruct")
        with open(exp_file, "r", encoding="utf-8") as f:
            exp_cfg = yaml.safe_load(f)
        exp_cfg["student_provenance"]["active_base_model"] = promoted_base
        exp_cfg["rewriter"]["name"] = promoted_base
        with open(exp_file, "w", encoding="utf-8") as f:
            yaml.dump(exp_cfg, f)
        print(f"Updated {exp_file} active_base_model to {promoted_base}")
else:
    print(f"Source adapter {source_adapter} not found. Please verify training output.")

# Register adapter in manifest and verify readiness gates
!python scripts/check_eval_readiness.py --register-adapter


## 9. Interactive Demonstration: Privacy Rewriting in Action
Test the fine-tuned rewriter model live on test prompts with demographic preambles to observe preamble pruning and natural generalization in real-time.

In [ ]:
# Cell 10: Live Inference Demonstration
# 1. Guard against broken multimodal extensions in Colab
!pip uninstall -y -q torchvision torchaudio 2>/dev/null || true

import sys
import torch
from pathlib import Path

# Clear cached modules to ensure the newest code is loaded in running Colab kernel
for mod in list(sys.modules.keys()):
    if mod.startswith("src.rewriter") or mod.startswith("peft"):
        del sys.modules[mod]

# Disable torchvision/torchaudio hooks inside transformers
try:
    import transformers.utils.import_utils as _iu
    _iu.is_torchvision_available = lambda: False
    _iu.is_torchaudio_available = lambda: False
except Exception:
    pass

from src.rewriter.inference import QwenRewriterInference

# 2. Auto-resolve the fine-tuned adapter path
candidate_paths = [
    Path("artifacts/rewriter_qlora"),
    Path("artifacts/rewriter_qlora_0.5b"),
    Path("/content/drive/MyDrive/InferenceGuard/rewriter_0.5b"),
]
resolved_adapter = None
for p in candidate_paths:
    if (p / "adapter_config.json").exists():
        resolved_adapter = str(p)
        break

print(f"Target adapter: {resolved_adapter or 'None (using heuristic fallback)'}")
rewriter = QwenRewriterInference(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    adapter_path=resolved_adapter,
    load_in_4bit=torch.cuda.is_available(),
)
print(f"Adapter active: {rewriter._is_adapter_loaded} (Base: {rewriter.resolved_base_model})")

# 3. Test prompts demonstrating bidirectional persona pruning (preambles & postambles)
test_prompts = [
    # Preamble (front):
    "I am a 62-year-old retired cardiologist living in Miami Beach. What is the latest guideline for SGLT2 inhibitors in heart failure with preserved ejection fraction?",
    # Postamble (back):
    "What is the latest guideline for SGLT2 inhibitors in heart failure with preserved ejection fraction? I am a 62-year-old retired cardiologist living in Miami Beach.",
    # Introductory clause (front):
    "As an 18-year-old freshman at MIT struggling with 18.01 single variable calculus, can you explain the epsilon-delta definition of a limit with an intuitive diagram?",
    # Trailing note (back):
    "Can you explain the epsilon-delta definition of a limit with an intuitive diagram? FYI, I am an 18-year-old freshman at MIT struggling with 18.01 calculus.",
    # Trailing parenthetical (back):
    "Can you help me optimize my 401(k) asset allocation between VOO and VXUS? (Software engineer in Seattle earning $180k)",
]

print("\n=== LIVE INFERENCE DEMO ===\n")
for orig in test_prompts:
    rewritten = rewriter.rewrite(orig)
    if isinstance(rewritten, tuple):
        rewritten = rewritten[0]
    print(f"ORIGINAL:\n{orig}\n")
    print(f"REWRITTEN:\n{rewritten}\n")
    print("-" * 70)


## 10. Comprehensive 50-Example Benchmark Evaluation
Run the complete 50-prompt test suite (`data/evaluation/benchmark_test_50.jsonl`) assessing:
1. **Co-occurring PII in Preambles**: Multiple demographic attributes (age, location, occupation, education, salary) clustered at prompt start.
2. **Trailing PII in Postambles**: Personal information introduced after the query (`FYI...`, `By the way...`, parentheticals).
3. **Essential Legal Jurisdictions**: State and local statutes (California, Texas, New York, Florida) that MUST be preserved.
4. **Clinical Patient Presentations**: Third-party patient case studies (infant, child, geriatric parameters) that MUST NOT be scrubbed.
5. **Hybrid Edge Cases**: Legal/medical questions with essential jurisdiction where the user ALSO appends personal demographics.

In [ ]:
# Cell 11: Run 50-Example Benchmark Suite and Generate Evaluation Report
!PYTHONPATH=. python scripts/evaluate_benchmark_50.py \
    --adapter_path artifacts/rewriter_qlora \
    --benchmark_file data/evaluation/benchmark_test_50.jsonl \
    --output_report reports/benchmark_50_results.json


In [ ]:
# Cell 12: View Full Benchmark Test Cases (Original vs Rewritten without Truncation)
import json
from pathlib import Path

report_file = Path("reports/benchmark_50_results.json")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"=== BENCHMARK REPORT (Total: {data.get('total_cases')}, Passed: {data.get('passed_cases')} [{data.get('pass_rate')*100:.1f}%]) ===\n")
    for tc in data.get("results", []):
        status = "[PASS]" if tc.get("passed") else "[FAIL]"
        print(f"{status} #{tc.get('id')} ({tc.get('category')}):")
        print(f"    ORIGINAL:  {tc.get('input')}")
        print(f"    REWRITTEN: {tc.get('rewritten')}")
        if tc.get("notes"):
            print(f"    ISSUES:    {', '.join(tc.get('notes'))}")
        print("-" * 70)
else:
    print("No benchmark report found. Run Cell 11 first!")


## 11. Launch InferenceGuard Interactive Web UI
Launch the FastAPI + Uvicorn server in the background and expose the web dashboard via Google Colab port forwarding or a public tunnel.


In [ ]:
# Cell 13: Launch InferenceGuard Web UI (Uvicorn + Colab Port Forwarding / Tunnel)
!pip install -q fastapi uvicorn pyngrok nest-asyncio
!pkill -f uvicorn 2>/dev/null || true

import time
import subprocess

# 1. Start Uvicorn backend in background
get_ipython().system_raw("uvicorn web.api:app --host 0.0.0.0 --port 8000 &")
print("Starting Uvicorn server on port 8000...")
time.sleep(5)

# 2. Method A (Recommended for Google Colab): Native Port Viewer
try:
    from google.colab.output import serve_kernel_port_as_window
    serve_kernel_port_as_window(8000, anchor_text="🚀 Click here to open InferenceGuard UI in a new tab")
    print("Google Colab port forwarding link active above!")
except Exception:
    pass

# 3. Method B: Localtunnel Public URL (Optional fallback)
try:
    import urllib.request
    colab_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()
    print(f"\nLocaltunnel IP Password: {colab_ip}")
    print("Launching Localtunnel (click link below and enter password if prompted):")
    !npx localtunnel --port 8000
except Exception as e:
    print(f"Tunnel note: {e}")
